## Cell 1 — Imports

In [1]:
import re
import time
from pathlib import Path

import cv2
import fitz  # PyMuPDF
import numpy as np
import pandas as pd
import pytesseract
from pytesseract import Output
import re

## Cell 2 — Configuration

In [2]:
import re

In [3]:
# CONFIGURATION

PDF_DPI = 200

MIN_EMBEDDED_IMAGE_AREA = 500 * 500

OCR_CONFIG = "--psm 6"

MIN_AMOUNT = 0.01

AMOUNT_COLUMN_MARGIN_X = 45

HEADER_UPSCALE = 2

# Matches "Amount", "Amount (Rs.)", "Amount(INR)", "Amt", etc.
AMOUNT_HEADER_RE = re.compile(r"amou?nt", re.IGNORECASE)


## Cell 3 — Amount Regex + Parser

In [4]:
# # AMOUNT PATTERN + PARSER

# AMOUNT_RE = re.compile(
#     r"""
#     ^[+-]?
#     (?:
#         \d{1,3}(?:,\d{3})+(?:\.\d{1,2})?
#         |
#         \d+(?:\.\d{1,2})?
#     )
#     $
#     """,
#     re.VERBOSE
# )


# def normalize_amount_token(text):
#     """Fix common OCR mistakes (O->0, I/l/|->1, S->5) and strip anything
#     that can't appear in an accounting number."""

#     if text is None:
#         return ""

#     s = str(text).strip()

#     for old, new in {
#         "O": "0", "o": "0",
#         "I": "1", "l": "1", "|": "1",
#         "S": "5",
#         " ": "", "\t": "", "\n": "", "\r": "",
#     }.items():
#         s = s.replace(old, new)

#     return re.sub(r"[^0-9,.\-+]", "", s)


# def parse_amount(text):
#     """
#     Convert OCR text into a signed float, or None if it isn't a valid
#     accounting amount.

#     Handles:
#         31,746.00      31,746.00-      31.746.00-  (OCR turned , into .)
#         31746.00       -31746.00       2,054.00-
#     """

#     s = normalize_amount_token(text)
#     if not s:
#         return None

#     negative = False

#     # SAP/accounting trailing minus: 31,746.00-
#     if s.endswith("-"):
#         negative = True
#         s = s[:-1]

#     if s.startswith("-"):
#         negative = True
#         s = s[1:]

#     if s.startswith("+"):
#         s = s[1:]

#     if not s:
#         return None

#     # OCR sometimes turns the thousands comma into a dot: 31.746.00
#     if s.count(".") > 1 and "," not in s:
#         parts = s.split(".")
#         if len(parts[-1]) in (1, 2) and all(p.isdigit() for p in parts):
#             s = ",".join(parts[:-1]) + "." + parts[-1]

#     if not AMOUNT_RE.fullmatch(s):
#         return None

#     try:
#         value = float(s.replace(",", ""))
#         if abs(value) < MIN_AMOUNT:
#             return None
#         return -value if negative else value
#     except (ValueError, TypeError):
#         return None


In [5]:
import re


# ============================================================
# AMOUNT PATTERN + PARSER
# ============================================================

AMOUNT_RE = re.compile(
    r"""
    ^[+-]?
    (?:
        # ----------------------------------------------------
        # Western / International grouping
        # 31,746.00
        # 3,174,600.00
        # 12,345,678.90
        # ----------------------------------------------------
        \d{1,3}(?:,\d{3})+(?:\.\d{1,2})?

        |

        # ----------------------------------------------------
        # Indian grouping
        # 1,00,000.00
        # 31,74,600.00
        # 3,17,46,000.00
        # 12,34,56,789.00
        # ----------------------------------------------------
        \d{1,3}(?:,\d{2})+,\d{3}(?:\.\d{1,2})?

        |

        # ----------------------------------------------------
        # Plain number
        # 31746.00
        # 31746000.00
        # ----------------------------------------------------
        \d+(?:\.\d{1,2})?
    )
    $
    """,
    re.VERBOSE,
)


def normalize_amount_token(text):
    """
    Normalize OCR text before amount parsing.

    Handles common OCR digit confusions:
        O / o -> 0
        I / l / | -> 1

    Preserves:
        digits
        comma
        decimal point
        plus/minus

    Examples:
        '31,746.00'       -> '31,746.00'
        '3,17,46,000.00'  -> '3,17,46,000.00'
        'O,746.OO'        -> '0,746.00'
        '31,746.00-'      -> '31,746.00-'
    """

    if text is None:
        return ""

    s = str(text).strip()

    # Remove all whitespace
    s = re.sub(r"\s+", "", s)

    # Common high-confidence OCR corrections
    s = s.translate(
        str.maketrans({
            "O": "0",
            "o": "0",
            "I": "1",
            "l": "1",
            "|": "1",
        })
    )

    # Keep only characters that can belong to an amount
    s = re.sub(r"[^0-9,.\-+]", "", s)

    return s


def parse_amount(text):
    """
    Convert OCR text into a signed float, or None if invalid.

    Supports:

        31,746.00
        31,746.00-
        -31,746.00

        31,74,600.00
        31,74,600.00-
        -31,74,600.00

        3,17,46,000.00

        31746.00
        31746000.00

        31.746.00-
        (OCR comma -> dot)

        2,054.00-

    Returns:
        float
        None
    """

    s = normalize_amount_token(text)

    if not s:
        return None

    negative = False

    # --------------------------------------------------------
    # Handle trailing minus
    # Example: 31,746.00-
    # --------------------------------------------------------
    if s.endswith("-"):
        negative = True
        s = s[:-1]

    # --------------------------------------------------------
    # Handle leading minus
    # Example: -31,746.00
    # --------------------------------------------------------
    if s.startswith("-"):
        negative = True
        s = s[1:]

    # --------------------------------------------------------
    # Handle leading plus
    # --------------------------------------------------------
    if s.startswith("+"):
        s = s[1:]

    if not s:
        return None

    # --------------------------------------------------------
    # OCR correction:
    #
    # 31.746.00
    #
    # Sometimes OCR converts:
    #
    # 31,746.00
    #
    # into:
    #
    # 31.746.00
    #
    # Convert it back to:
    #
    # 31,746.00
    # --------------------------------------------------------
    if s.count(".") > 1 and "," not in s:

        parts = s.split(".")

        if (
            len(parts[-1]) in (1, 2)
            and all(part.isdigit() for part in parts)
        ):
            s = ",".join(parts[:-1]) + "." + parts[-1]

    # --------------------------------------------------------
    # Validate amount format
    # --------------------------------------------------------
    if not AMOUNT_RE.fullmatch(s):
        return None

    # --------------------------------------------------------
    # Convert to numeric value
    # --------------------------------------------------------
    try:
        value = float(s.replace(",", ""))

        # Business-level minimum amount validation
        if abs(value) < MIN_AMOUNT:
            return None

        return -value if negative else value

    except (ValueError, TypeError):
        return None

## Cell 4 — Test Parser

Run on its own first. If every line looks right, the parser is safe to trust for the rest of
the notebook.

In [6]:
# ============================================================
# PARSER TEST
# ============================================================

test_values = [
    "31,746.00", "31,746.00-", "31.746.00-", "31746.00",
    "2,054.00-", "41,085.00", "O1,746.00", "43,139",
    "1,234,567.89", "-31746.00", "+31746.00", "abc", "", "0.00","3,17,46,000.00","31.746.000.00"
]

for value in test_values:
    print(f"{value!r:<18} -> {parse_amount(value)}")


'31,746.00'        -> 31746.0
'31,746.00-'       -> -31746.0
'31.746.00-'       -> -31746.0
'31746.00'         -> 31746.0
'2,054.00-'        -> -2054.0
'41,085.00'        -> 41085.0
'O1,746.00'        -> 1746.0
'43,139'           -> 43139.0
'1,234,567.89'     -> 1234567.89
'-31746.00'        -> -31746.0
'+31746.00'        -> 31746.0
'abc'              -> None
''                 -> None
'0.00'             -> None
'3,17,46,000.00'   -> 31746000.0
'31.746.000.00'    -> 31746000.0


## Cell 5 — PDF Rendering (with embedded-image fast path)

In [7]:
# PDF -> IMAGES

def render_pdf(path, dpi=PDF_DPI):
    """
    Convert PDF pages into OpenCV BGR images.

    Priority per page:
    1. Extract a large embedded image directly (sharper and much faster
       than re-rendering a scanned/printed page).
    2. Otherwise render the page at `dpi`.
    """

    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Input file not found: {path}")

    pages = []

    try:
        doc = fitz.open(str(path))

        for page_index, page in enumerate(doc):
            page_no = page_index + 1

            best_xref, best_area = None, 0

            for img_info in page.get_images(full=True):
                xref = img_info[0]
                try:
                    pix = fitz.Pixmap(doc, xref)
                    area = pix.width * pix.height
                    if area > best_area and area >= MIN_EMBEDDED_IMAGE_AREA:
                        best_area, best_xref = area, xref
                except Exception:
                    continue

            if best_xref is not None:
                pix = fitz.Pixmap(doc, best_xref)

                if pix.colorspace and pix.colorspace.n not in (1, 3):
                    pix = fitz.Pixmap(fitz.csRGB, pix)  # normalize CMYK etc.

                if pix.n >= 4:
                    arr = np.frombuffer(pix.samples, dtype=np.uint8).reshape(
                        pix.height, pix.width, pix.n)
                    img = cv2.cvtColor(arr[:, :, :4], cv2.COLOR_RGBA2BGR)
                elif pix.n == 3:
                    arr = np.frombuffer(pix.samples, dtype=np.uint8).reshape(
                        pix.height, pix.width, 3)
                    img = cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)
                else:
                    arr = np.frombuffer(pix.samples, dtype=np.uint8).reshape(
                        pix.height, pix.width)
                    img = cv2.cvtColor(arr, cv2.COLOR_GRAY2BGR)

                pages.append((page_no, img))
                continue

            # Fallback: render the full page (alpha=False -> always 3-channel)
            scale = dpi / 72.0
            pix = page.get_pixmap(matrix=fitz.Matrix(scale, scale), alpha=False)
            arr = np.frombuffer(pix.samples, dtype=np.uint8).reshape(
                pix.height, pix.width, 3)
            img = cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)
            pages.append((page_no, img))

        doc.close()
        return pages

    except Exception as e:
        raise RuntimeError(f"Unable to process PDF '{path.name}': {e}") from e


## Cell 6 — Input Loader (PDF / PNG / JPG / JPEG / WEBP)

In [8]:
# INPUT LOADER

SUPPORTED_EXTENSIONS = {".pdf", ".png", ".jpg", ".jpeg", ".webp"}


def load_input(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Input file does not exist: {path}")

    if path.suffix.lower() not in SUPPORTED_EXTENSIONS:
        raise ValueError(f"Unsupported file type: {path.suffix}")

    if path.suffix.lower() == ".pdf":
        pages = render_pdf(path)
        if not pages:
            raise RuntimeError(f"No readable pages found in: {path.name}")
        return pages

    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        raise RuntimeError(f"Unable to read image: {path.name}")
    return [(1, img)]


## Cell 7 — Blank Page Detection

In [9]:
# BLANK PAGE DETECTION

def is_blank_page(img):
    if img is None:
        return True
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return np.mean(gray) > 245 and np.mean(gray < 245) < 0.02


def remove_blank_pages(pages):
    valid = []
    for page_no, img in pages:
        try:
            if not is_blank_page(img):
                valid.append((page_no, img))
        except Exception as e:
            print(f"Warning: blank-page check failed for page {page_no}: {e}")
            valid.append((page_no, img))  # keep rather than risk dropping content
    return valid


## Cell 8 — OCR Helper

In [10]:
# OCR

def run_ocr(img, config=OCR_CONFIG):
    if img is None or img.size == 0:
        return ""
    try:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
        gray = cv2.resize(gray, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
        processed = cv2.threshold(
            gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )[1]
        return pytesseract.image_to_string(processed, config=config)
    except Exception as e:
        print(f"OCR warning: {e}")
        return ""


## Cell 9 — Amount Header Detection + Column Extraction

`find_amount_header` runs on an **upscaled copy** of the page — this is the fix that made the
previously-failing SAP screenshot work: at native resolution Tesseract silently dropped the
header word entirely.

`extract_from_page` then crops a tight vertical strip around that header and OCRs only the
strip, line by line. A whole-page OCR fallback only kicks in if no header can be found at all —
verified to be less reliable on dense screenshots, so it's a last resort, not the main path.

In [11]:
# AMOUNT COLUMN DETECTION + PER-PAGE EXTRACTION

def find_amount_header(img):
    """Locate the "Amount" header's bounding box (topmost match wins)."""

    try:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
        scaled = cv2.resize(
            gray, None, fx=HEADER_UPSCALE, fy=HEADER_UPSCALE,
            interpolation=cv2.INTER_CUBIC
        )
        data = pytesseract.image_to_data(scaled, output_type=Output.DICT)
    except Exception as e:
        print(f"Header detection OCR warning: {e}")
        return None

    best = None
    for i in range(len(data.get("text", []))):
        word = (data["text"][i] or "").strip()
        if not word or not AMOUNT_HEADER_RE.search(word):
            continue

        box = {
            "left": data["left"][i] // HEADER_UPSCALE,
            "top": data["top"][i] // HEADER_UPSCALE,
            "width": data["width"][i] // HEADER_UPSCALE,
            "height": data["height"][i] // HEADER_UPSCALE,
        }
        if best is None or box["top"] < best["top"]:
            best = box

    return best


def crop_amount_column(img, header_box, margin_x=AMOUNT_COLUMN_MARGIN_X):
    """Crop a full-height vertical strip around the detected header."""
    h, w = img.shape[:2]
    left = max(0, header_box["left"] - margin_x)
    right = min(w, header_box["left"] + header_box["width"] + margin_x)
    top = max(0, header_box["top"] - 5)
    return img[top:h, left:right]
# The function returns a new, cropped NumPy array (image).1. Visual Dimensions of the Output:It is a narrow, vertical strip that is 120 pixels wide (\(850 - 730\)) and 1305 pixels high (\(1500 - 195\)).2. Visual Content Inside this Cropped Box:Because it starts at pixel height 195 (just above the header), the returned image strip contains:At the very top (Y-pixels 0 to 25): The text word "Amount".Directly below it: Any white space, divider lines, and every single price listed in that column all the way to the bottom of the invoice page (e.g., $150.00, $2,450.00, etc.).




# Amount-shaped token finder, used as a per-line safety net (a line with
# stray characters around the number) and as the whole-page fallback when
# no header can be located at all. Requires a "," or "." in the token so
# invoice numbers / dates / quantities (bare digit runs) aren't picked up.
_CANDIDATE_RE = re.compile(
    r"""(?<![0-9])[+-]?(?:\d{1,3}(?:[,.]\d{3})*|\d+)(?:[.,]\d{1,2})?-?(?![0-9])""",
    re.VERBOSE
)


def extract_amount_candidates_from_text(text):
    if not text:
        return []
    candidates = []
    for token in _CANDIDATE_RE.findall(text):
        if "," not in token and "." not in token:
            continue
        value = parse_amount(token)
        if value is not None:
            candidates.append({"raw": token, "value": value})
    return candidates


def extract_from_page(img, page_no):
    """
    Returns: {"rows": [{"page", "raw", "value"}, ...], "column_detected": bool}
    """

    rows = []
    header_box = find_amount_header(img)

    if header_box is not None:

        crop = crop_amount_column(img, header_box)
        text = run_ocr(crop, config=OCR_CONFIG)

        for line in text.splitlines():
            line = line.strip()
            if not line:
                continue

            # Skip the header line itself
            if AMOUNT_HEADER_RE.fullmatch(re.sub(r"[^A-Za-z]", "", line)):
                continue

            if parse_amount(line) is not None:
                rows.append({"page": page_no, "raw": line, "value": line})
                continue

            # Line has extra noise around the number - pull the token out
            for candidate in extract_amount_candidates_from_text(line):
                rows.append({"page": page_no, "raw": line, "value": candidate["raw"]})

        return {"rows": rows, "column_detected": True}

    # Fallback: no "Amount" header found anywhere on the page
    text = run_ocr(img, config=OCR_CONFIG)
    for candidate in extract_amount_candidates_from_text(text):
        rows.append({"page": page_no, "raw": candidate["raw"], "value": candidate["raw"]})

    return {"rows": rows, "column_detected": False}


## Cell 10 — Complete Extraction Engine

Ties every stage together, times each stage, and never crashes: any failure returns a
structured `{"success": False, "error": ...}` result instead of raising.

In [12]:
# COMPLETE EXTRACTION ENGINE

def extract_max_amount(path, return_debug=False):

    path = Path(path)
    total_start = time.perf_counter()
    timing = {
        "load_input": 0.0, "blank_page_detection": 0.0,
        "amount_extraction": 0.0, "amount_selection": 0.0, "total": 0.0,
    }

    def fail(error, message, rows=None, **extra):
        timing["total"] = time.perf_counter() - total_start
        return {
            "success": False, "amount": None, "error": error, "message": message,
            "rows": rows or [], "timing": timing, **extra,
        }

    try:
        # 1. LOAD INPUT
        start = time.perf_counter()
        pages = load_input(path)
        timing["load_input"] = time.perf_counter() - start

        if not pages:
            return fail("NO_PAGES", f"No pages found in {path.name}")

        # 2. REMOVE BLANK PAGES
        start = time.perf_counter()
        non_blank_pages = remove_blank_pages(pages)
        timing["blank_page_detection"] = time.perf_counter() - start

        if not non_blank_pages:
            return fail("ALL_PAGES_BLANK", f"All pages are blank in {path.name}")

        # 3. AMOUNT EXTRACTION
        start = time.perf_counter()
        all_rows, page_errors, any_column_detected = [], [], False

        for page_no, img in non_blank_pages:
            try:
                result = extract_from_page(img, page_no)
                if result.get("column_detected"):
                    any_column_detected = True
                all_rows.extend(result.get("rows", []))
            except Exception as e:
                page_errors.append({"page": page_no, "error": str(e)})
                print(f"Warning: extraction failed on page {page_no}: {e}")

        timing["amount_extraction"] = time.perf_counter() - start

        if not all_rows:
            return fail(
                "NO_AMOUNT_DETECTED", f"No valid Amount values found in: {path.name}",
                pages_processed=len(non_blank_pages),
                column_detected=any_column_detected, page_errors=page_errors,
            )

        # 4. FINAL AMOUNT SELECTION
        start = time.perf_counter()
        values = [parse_amount(r.get("value")) for r in all_rows if isinstance(r, dict)]
        values = [v for v in values if v is not None]
        timing["amount_selection"] = time.perf_counter() - start

        if not values:
            return fail(
                "NO_VALID_AMOUNT",
                f"Amount column detected but no valid numeric amount could be "
                f"parsed in: {path.name}",
                rows=all_rows, pages_processed=len(non_blank_pages),
                column_detected=any_column_detected, page_errors=page_errors,
            )

        # 5. SELECT MAXIMUM (largest positive; largest magnitude if all negative)
        positive_values = [v for v in values if v > 0]
        final_amount = max(positive_values) if positive_values else max(values, key=abs)

        timing["total"] = time.perf_counter() - total_start

        result = {
            "success": True, "amount": final_amount, "error": None,
            "message": "Amount extracted successfully",
            "rows": all_rows, "pages_processed": len(non_blank_pages),
            "column_detected": any_column_detected, "page_errors": page_errors,
            "timing": timing,
        }

        return result if return_debug else final_amount

    except FileNotFoundError as e:
        return fail("FILE_NOT_FOUND", str(e))
    except ValueError as e:
        return fail("INVALID_INPUT", str(e))
    except Exception as e:
        return fail("ENGINE_ERROR", str(e))


## Cell 11 — Run the Engine

Set `path` to your PDF/PNG/JPG/WEBP file, then run this cell.

In [13]:


# print("Please upload your PDF file:")
# uploaded_pdf = files.upload()

# if not uploaded_pdf:
#     raise RuntimeError("No file was uploaded.")

# pdf_filename = list(uploaded_pdf.keys())[0]
PDF_PATH = r"D:\xfcatr\FASTAPI-OCR-API\data\temp_1788493964707_wu4epu_1788493960711_5100114841 - DN - MAA BHAGWATI-1 1.webp"

print("\n--- Path Set ---")
print(f"PDF_PATH: {PDF_PATH}")

path = PDF_PATH  # <-- the uploaded file

result = extract_max_amount(path, return_debug=True)

print("=" * 60)
print("AMOUNT EXTRACTION RESULT")
print("=" * 60)

if result["success"]:
    print("Status          : SUCCESS")
    print("Amount          :", f"{result['amount']:,.2f}")
else:
    print("Status          : FAILED")
    print("Error           :", result["error"])
    print("Message         :", result["message"])

print("Execution Time  :", f"{result['timing']['total']:.4f} sec")
print("=" * 60)


--- Path Set ---
PDF_PATH: D:\xfcatr\FASTAPI-OCR-API\data\temp_1788493964707_wu4epu_1788493960711_5100114841 - DN - MAA BHAGWATI-1 1.webp
AMOUNT EXTRACTION RESULT
Status          : SUCCESS
Amount          : 43,139.00
Execution Time  : 3.0924 sec


## Cell 12 — Timing Breakdown

In [14]:
timing = result["timing"]

print("ENGINE EXECUTION BREAKDOWN")
print("-" * 45)
for name, value in timing.items():
    print(f"{name:<30}: {value:.4f} sec")


ENGINE EXECUTION BREAKDOWN
---------------------------------------------
load_input                    : 0.0341 sec
blank_page_detection          : 0.0191 sec
amount_extraction             : 3.0391 sec
amount_selection              : 0.0000 sec
total                         : 3.0924 sec


In [18]:


IMAGE_PATH = r"D:\xfcatr\FASTAPI-OCR-API\data\100250858-MARUTHI-DN 1.pdf"
print(IMAGE_PATH)
result = extract_max_amount(path, return_debug=True)

print("=" * 60)
print("AMOUNT EXTRACTION RESULT")
print("=" * 60)

if result["success"]:
    print("Status          : SUCCESS")
    print("Amount          :", f"{result['amount']:,.2f}")
else:
    print("Status          : FAILED")
    print("Error           :", result["error"])
    print("Message         :", result["message"])

print("Execution Time  :", f"{result['timing']['total']:.4f} sec")
print("=" * 60)

D:\xfcatr\FASTAPI-OCR-API\data\100250858-MARUTHI-DN 1.pdf
AMOUNT EXTRACTION RESULT
Status          : SUCCESS
Amount          : 43,139.00
Execution Time  : 2.1230 sec


In [ ]:
# RUN ENGINE
from pathlib import Path
from google.colab import files

print("Please upload your PDF file:")
uploaded_pdf = files.upload()

if not uploaded_pdf:
    raise RuntimeError("No file was uploaded.")

pdf_filename = list(uploaded_pdf.keys())[0]
PDF_PATH = Path(pdf_filename)

print("\n--- Path Set ---")
print(f"PDF_PATH: {PDF_PATH}")
# <-- the uploaded file

result = extract_max_amount(path, return_debug=True)

print("=" * 60)
print("AMOUNT EXTRACTION RESULT")
print("=" * 60)

if result["success"]:
    print("Status          : SUCCESS")
    print("Amount          :", f"{result['amount']:,.2f}")
else:
    print("Status          : FAILED")
    print("Error           :", result["error"])
    print("Message         :", result["message"])

print("Execution Time  :", f"{result['timing']['total']:.4f} sec")
print("=" * 60)

Please upload your PDF file:


Saving 100250858-MARUTHI-DN 1.pdf to 100250858-MARUTHI-DN 1 (3).pdf

--- Path Set ---
PDF_PATH: 100250858-MARUTHI-DN 1 (3).pdf
AMOUNT EXTRACTION RESULT
Status          : SUCCESS
Amount          : 31,746.00
Execution Time  : 3.5291 sec


## Cell 13 — (Optional) Inspect All Detected Rows

Useful when the wrong value gets picked, or nothing is found.

In [20]:
# if result.get("rows"):
#     df = pd.DataFrame(result["rows"])
#     df["parsed_value"] = df["value"].apply(parse_amount)
#     display(df)
# else:
#     print("No rows were detected for this document.")

# print("\nColumn header detected on at least one page:", result.get("column_detected"))
# if result.get("page_errors"):
#     print("Per-page errors:", result["page_errors"])
